In [13]:
# Celda 1 - Librerias  [V0.5]
import cv2
import numpy as np
import time
import math
import datetime
import customtkinter as ctk
from PIL import Image

# Librerias para Optimizador DEAP
import random
import operator
import pandas as pd
from deap import base, creator, tools, gp, algorithms

ctk.set_appearance_mode('dark')
print('Celda 1 V0.5: Librerias base y DEAP cargadas.')

Celda 1 V0.5: Librerias base y DEAP cargadas.


In [14]:
# Celda 2 - Motor de vision con Mallas YOLO y HoughCircles  [V0.5]
from pygrabber.dshow_graph import FilterGraph


def detectar_camaras_sistema():
    try:
        graph = FilterGraph()
        return [(i, n) for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar camaras: {e}')
        return []

RANGO_ROJO_1 = (np.array([  0, 100,  60]), np.array([ 12, 255, 255]))
RANGO_ROJO_2 = (np.array([168, 100,  60]), np.array([180, 255, 255]))
RANGO_NEGRO  = (np.array([  0,   0,   0]), np.array([180, 255,  60]))
RANGO_BLANCO = (np.array([  0,   0, 170]), np.array([180,  45, 255]))

HOUGH_DP       = 1.2
HOUGH_MINDIST  = 60
HOUGH_PARAM1   = 120
HOUGH_PARAM2   = 38
HOUGH_MINR     = 35
HOUGH_MAXR     = 320

FRACCION_COLOR_MIN = 0.20
UMBRAL_MISMO_OBJETO = 0.82

def clasificar_color_circulo(hsv_frame, cx, cy, radio):
    h, w = hsv_frame.shape[:2]
    mascara = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mascara, (cx, cy), radio, 255, -1)
    total_px = cv2.countNonZero(mascara)
    if total_px == 0: return None

    m_rojo = cv2.add(cv2.inRange(hsv_frame, *RANGO_ROJO_1),
                     cv2.inRange(hsv_frame, *RANGO_ROJO_2))
    m_negro  = cv2.inRange(hsv_frame, *RANGO_NEGRO)
    m_blanco = cv2.inRange(hsv_frame, *RANGO_BLANCO)

    fracs = {
        'Rojo':   cv2.countNonZero(cv2.bitwise_and(m_rojo,  mascara)) / total_px,
        'Negro':  cv2.countNonZero(cv2.bitwise_and(m_negro, mascara)) / total_px,
        'Blanco': cv2.countNonZero(cv2.bitwise_and(m_blanco,mascara)) / total_px,
    }
    mejor = max(fracs, key=fracs.get)
    if fracs[mejor] >= FRACCION_COLOR_MIN:
        return mejor
    return None

def calcular_huella_hsv(hsv_frame, cx, cy, radio):
    h, w = hsv_frame.shape[:2]
    mascara = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mascara, (cx, cy), radio, 255, -1)
    if cv2.countNonZero(mascara) < 50: return None
    hist = cv2.calcHist([hsv_frame], [0, 1], mascara,
                        [30, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist

def es_mismo_objeto(huella_nueva, huellas_guardadas):
    if huella_nueva is None or not huellas_guardadas: return False
    for ref in huellas_guardadas:
        if cv2.compareHist(huella_nueva, ref, cv2.HISTCMP_CORREL) >= UMBRAL_MISMO_OBJETO:
            return True
    return False

def get_color_bgr(nombre):
    n = nombre.lower()
    if 'rojo'   in n or 'red'   in n: return (0,   0, 220)
    if 'blanco' in n or 'white' in n: return (200, 200, 200)
    if 'negro'  in n or 'black' in n: return (80,  80,  80)
    return (0, 220, 220)

def estimar_z(radio_px, frame_shape):
    frac = (math.pi * radio_px * radio_px) / (frame_shape[0] * frame_shape[1])
    return round(max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01))), 2)

class TrackerCirculo:
    def __init__(self, alpha=0.35, frames_conf=2, frames_perdida=5):
        self.alpha          = alpha
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.suave       = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, deteccion):
        if deteccion is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible = False
                self.suave   = None
            return self.suave if self.visible else None

        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)
        if self.conteo_det < self.frames_conf:
            return None
        if self.suave is None:
            self.suave   = tuple(float(v) for v in deteccion)
            self.visible = True
            return tuple(int(round(v)) for v in self.suave)

        self.visible = True
        a = self.alpha
        self.suave = tuple(a * d + (1 - a) * s for d, s in zip(deteccion, self.suave))
        return tuple(int(round(v)) for v in self.suave)

def procesar_frame_vision(frame, temporizadores, trackers, callback_objeto=None, yolo_model=None):
    t_act       = time.time()
    hay_objetos = False
    fh, fw      = frame.shape[:2]

    # ==================================================================
    # MODO YOLO CON MALLAS (SEGMENTACION DE ALTA CALIDAD)
    # ==================================================================
    if yolo_model is not None:
        results = yolo_model.predict(frame, conf=0.45, verbose=False)
        yolo_dets = {}

        if results and results[0].masks is not None:
            boxes = results[0].boxes
            masks = results[0].masks.xy
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i])
                nombre = yolo_model.names[cls_id]
                x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy()
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)
                r  = int(max(x2 - x1, y2 - y1) / 2)
                conf = float(boxes.conf[i])
                pts = np.array(masks[i], dtype=np.int32)
                yolo_dets.setdefault(nombre, []).append((cx, cy, r, conf, pts))

        for nombre in yolo_dets:
            if nombre not in trackers:
                trackers[nombre] = TrackerCirculo()
                temporizadores[nombre] = 0.0

        for nombre, tr in list(trackers.items()):
            det, pts = None, None
            if nombre in yolo_dets:
                mejor = max(yolo_dets[nombre], key=lambda d: d[3])
                det = mejor[:3]
                pts = mejor[4]

            result = tr.actualizar(det)
            if result is None:
                temporizadores[nombre] = 0.0
                continue

            hay_objetos = True
            cx, cy, r = result
            if temporizadores[nombre] == 0.0:
                temporizadores[nombre] = t_act

            color_bgr = get_color_bgr(nombre)
            x_norm = round(cx / fw * 2 - 1, 2)
            y_norm = round(1 - cy / fh * 2, 2)
            z_est  = estimar_z(r, frame.shape)

            # Dibujar Malla de Alta Calidad
            if pts is not None:
                overlay = frame.copy()
                cv2.fillPoly(overlay, [pts], color_bgr)
                cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
                cv2.polylines(frame, [pts], True, color_bgr, 2)
            else:
                cv2.circle(frame, (cx, cy), r, color_bgr, 2)

            lbl = f'YOLO(malla): {nombre}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'
            label_y = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx - r, label_y - th - 4), (cx - r + tw + 4, label_y + 4), (20, 20, 20), -1)
            cv2.putText(frame, lbl, (cx - r + 2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx - r, cy2 - ch - 2), (cx - r + cw + 4, cy2 + 2), (20, 20, 20), -1)
            cv2.putText(frame, lbl_c, (cx - r + 2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 210, 210), 1)

            if callback_objeto is not None:
                callback_objeto(nombre, x_norm, y_norm, z_est, None)

        return frame, hay_objetos

    # ==================================================================
    # MODO HOUGH CIRCLES + HSV
    # ==================================================================
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (9, 9), 2)
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    raw  = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=HOUGH_DP,
                            minDist=HOUGH_MINDIST, param1=HOUGH_PARAM1, param2=HOUGH_PARAM2,
                            minRadius=HOUGH_MINR, maxRadius=HOUGH_MAXR)
    detecciones_por_color = {}
    if raw is not None:
        for cx, cy, r in np.round(raw[0, :]).astype(int):
            nombre = clasificar_color_circulo(hsv, cx, cy, r)
            if nombre is not None:
                if nombre not in detecciones_por_color or r > detecciones_por_color[nombre][2]:
                    detecciones_por_color[nombre] = (cx, cy, r)

    for nombre in ('Rojo', 'Negro', 'Blanco'):
        if nombre not in trackers:
            trackers[nombre] = TrackerCirculo()
            temporizadores[nombre] = 0.0

    for nombre, tr in trackers.items():
        result = tr.actualizar(detecciones_por_color.get(nombre, None))
        if result is None:
            temporizadores[nombre] = 0.0
            continue

        hay_objetos = True
        cx, cy, r = result
        if temporizadores[nombre] == 0.0:
            temporizadores[nombre] = t_act

        color_bgr = get_color_bgr(nombre)
        x_norm = round(cx / fw * 2 - 1, 2)
        y_norm = round(1 - cy / fh * 2, 2)
        z_est  = estimar_z(r, frame.shape)

        cv2.circle(frame, (cx, cy), r, color_bgr, 2)
        cv2.circle(frame, (cx, cy), 3, color_bgr, -1)

        if (t_act - temporizadores[nombre]) <= 3.0:
            cv2.putText(frame, 'Calibrando...', (cx - r, max(18, cy - r - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_bgr, 1)
        else:
            lbl = f'Hough: {nombre}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'
            label_y = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx - r, label_y - th - 4), (cx - r + tw + 4, label_y + 4), (20, 20, 20), -1)
            cv2.putText(frame, lbl, (cx - r + 2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx - r, cy2 - ch - 2), (cx - r + cw + 4, cy2 + 2), (20, 20, 20), -1)
            cv2.putText(frame, lbl_c, (cx - r + 2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 210, 210), 1)

            if callback_objeto is not None:
                huella = calcular_huella_hsv(hsv, cx, cy, r)
                callback_objeto(nombre, x_norm, y_norm, z_est, huella)

    return frame, hay_objetos

print('Celda 2 V0.5: Mallas YOLO y HoughCircles listos.')

Celda 2 V0.5: Mallas YOLO y HoughCircles listos.


In [15]:
# Celda 3 - UI con limite de registros [V0.5]
REINICIO_INACTIVO  = 0
REINICIO_ESPERANDO = 1
REINICIO_LISTO     = 2

class EscanerApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title('Deteccion de Pelotas  -  V0.5')
        self.geometry('1380x860')
        self.configure(fg_color='#1a1d2e')

        self.cap            = None
        self.escaneando     = False
        self.temporizadores = {'Rojo': 0.0, 'Blanco': 0.0, 'Negro': 0.0}
        self.trackers       = {
            'Rojo':   TrackerCirculo(),
            'Blanco': TrackerCirculo(),
            'Negro':  TrackerCirculo(),
        }
        self.huellas_memoria   = {'Rojo': [], 'Blanco': [], 'Negro': []}
        self.historial_objetos = []
        self.ultimo_intento    = {}
        self.COOLDOWN_INTENTO  = 1.5
        self.MAX_REGISTROS_POR_COLOR = 2  # <--- LIMITE ESTRICTO DE 2 REGISTROS

        self.modo_reinicio     = REINICIO_INACTIVO
        self.t_escena_vacia    = None
        self.frames_sin_objeto = 0
        self.FRAMES_VACIA_MIN  = 8

        self.yolo_model = None
        self._cargar_yolo()

        self._construir_encabezado()
        self._construir_panel_config()
        self._construir_panel_video()

    def _cargar_yolo(self):
        import os
        from ultralytics import YOLO
        model_path = os.path.join(os.getcwd(), 'models', 'entrenamiento_pelotas', 'weights', 'best.pt')
        if os.path.exists(model_path):
            try:
                self.yolo_model = YOLO(model_path)
                print(f'Modelo YOLO cargado: {model_path}')
            except Exception as e:
                print(f'Error YOLO: {e}. Modo HoughCircles activo.')
        else:
            print('Sin modelo YOLO. Modo HoughCircles activo.')

    def _construir_encabezado(self):
        modo = 'YOLO (Mallas)' if self.yolo_model else 'HoughCircles'
        self.lbl_titulo = ctk.CTkLabel(self, text=f'CONFIGURACION DE CAMARA  ({modo})',
                                       font=('Helvetica', 22, 'bold'), text_color='#e8eaf0')
        self.lbl_titulo.pack(pady=(28, 12))

    def _construir_panel_config(self):
        self.btn_detectar = ctk.CTkButton(self, text='DETECTAR CAMARAS',
                                          fg_color='#3498db', hover_color='#2980b9',
                                          font=('Helvetica', 14, 'bold'), command=self._accion_buscar)
        self.btn_detectar.pack(pady=10)
        self.lbl_estado = ctk.CTkLabel(self, text='Haz clic para buscar dispositivos', text_color='#a0aab5', font=('Helvetica', 12))
        self.lbl_estado.pack(pady=(0, 20))
        self.lbl_lista_tit = ctk.CTkLabel(self, text='Camaras Disponibles:', font=('Helvetica', 14, 'bold'), text_color='#ffffff')
        self.lbl_lista_tit.pack(anchor='w', padx=150)
        self.frame_lista = ctk.CTkScrollableFrame(self, fg_color='#2e3548', width=700, height=150,
                                                  corner_radius=8, border_width=1, border_color='#4a5568')
        self.frame_lista.pack(pady=5, padx=150)
        self.indice_sel = ctk.StringVar(value='-1')
        self.lbl_pie = ctk.CTkLabel(self, text='Ninguna camara seleccionada', text_color='#a0aab5', font=('Helvetica', 12))
        self.lbl_pie.pack(anchor='w', padx=150)
        self.btn_iniciar = ctk.CTkButton(self, text='INICIAR DETECCION', fg_color='#3b8ed0', hover_color='#296899',
                                         font=('Helvetica', 16, 'bold'), width=250, height=45,
                                         state='disabled', command=self._iniciar_camara)
        self.btn_iniciar.pack(pady=(40, 0))

    def _construir_panel_video(self):
        self.frame_monitor   = ctk.CTkFrame(self, fg_color='transparent')
        self.frame_video_col = ctk.CTkFrame(self.frame_monitor, fg_color='transparent')
        self.frame_video_col.pack(side='left', fill='both', expand=True)
        self.lbl_video = ctk.CTkLabel(self.frame_video_col, text='')
        self.lbl_video.pack(pady=10, padx=10)
        self.frame_reinicio_barra = ctk.CTkFrame(self.frame_video_col, fg_color='#0d1117', corner_radius=8, height=46)
        self.frame_reinicio_barra.pack(fill='x', padx=12, pady=(0, 4))
        self.frame_reinicio_barra.pack_propagate(False)
        self.lbl_reinicio_estado = ctk.CTkLabel(self.frame_reinicio_barra, text='', font=('Helvetica', 11, 'bold'), text_color='#556070')
        self.lbl_reinicio_estado.pack(side='left', padx=12)
        self.barra_progreso = ctk.CTkProgressBar(self.frame_reinicio_barra, width=320, height=12, fg_color='#1e2535', progress_color='#f39c12')
        self.barra_progreso.set(0)
        self.barra_progreso.pack(side='right', padx=12)
        self.frame_controles = ctk.CTkFrame(self.frame_video_col, fg_color='transparent')
        self.frame_controles.pack(pady=8)
        self.btn_escaneo = ctk.CTkButton(self.frame_controles, text='Iniciar Escaneo', font=('Helvetica', 13, 'bold'),
                                         fg_color='#f39c12', hover_color='#d68910', command=self._toggle_escaneo)
        self.btn_escaneo.grid(row=0, column=0, padx=10)
        self.btn_reinicio = ctk.CTkButton(self.frame_controles, text='Reinicio', font=('Helvetica', 13, 'bold'),
                                          fg_color='#8e44ad', hover_color='#6c3483', width=140, command=self._iniciar_modo_reinicio, state='disabled')
        self.btn_reinicio.grid(row=0, column=1, padx=10)
        self.btn_limpiar_todo = ctk.CTkButton(self.frame_controles, text='Limpiar todo', font=('Helvetica', 13, 'bold'),
                                              fg_color='#c0392b', hover_color='#96281b', width=140, command=self._limpiar_todo)
        self.btn_limpiar_todo.grid(row=0, column=2, padx=10)
        self.btn_detener = ctk.CTkButton(self.frame_controles, text='Detener Camara', font=('Helvetica', 13, 'bold'),
                                         fg_color='#e74c3c', hover_color='#c0392b', command=self._apagar_hardware)
        self.btn_detener.grid(row=0, column=3, padx=10)
        self.frame_historial = ctk.CTkFrame(self.frame_monitor, fg_color='#20253a', width=300, corner_radius=12, border_width=1, border_color='#343d5c')
        self.frame_historial.pack(side='right', fill='y', padx=(0, 12), pady=10)
        self.frame_historial.pack_propagate(False)
        encab = ctk.CTkFrame(self.frame_historial, fg_color='#151a2d', corner_radius=10)
        encab.pack(fill='x', padx=8, pady=(8, 4))
        ctk.CTkLabel(encab, text='PELOTAS DETECTADAS', font=('Helvetica', 12, 'bold'), text_color='#c8d0e0').pack(pady=(8, 2))
        self.lbl_conteo = ctk.CTkLabel(encab, text='Total: 0  |  En memoria: 0', font=('Helvetica', 9), text_color='#7a8aaa')
        self.lbl_conteo.pack(pady=(0, 8))
        self.lista_scroll = ctk.CTkScrollableFrame(self.frame_historial, fg_color='#1a1f33', corner_radius=8)
        self.lista_scroll.pack(fill='both', expand=True, padx=8, pady=(0, 8))

    def _iniciar_modo_reinicio(self):
        if self.modo_reinicio != REINICIO_INACTIVO:
            self._cancelar_reinicio()
            return
        self.modo_reinicio = REINICIO_ESPERANDO
        self.t_escena_vacia = None
        self.frames_sin_objeto = 0
        self.barra_progreso.set(0)
        self.btn_reinicio.configure(text='Cancelar reinicio', fg_color='#6c3483', hover_color='#4a235a')
        self._actualizar_barra_reinicio()

    def _cancelar_reinicio(self):
        self.modo_reinicio = REINICIO_INACTIVO
        self.t_escena_vacia = None
        self.frames_sin_objeto = 0
        self.barra_progreso.set(0)
        self.btn_reinicio.configure(text='Reinicio', fg_color='#8e44ad', hover_color='#6c3483')
        self.lbl_reinicio_estado.configure(text='Reinicio cancelado.', text_color='#e74c3c')
        self.after(2000, lambda: self.lbl_reinicio_estado.configure(text=''))

    def _procesar_estado_reinicio(self, hay_objetos):
        if self.modo_reinicio == REINICIO_INACTIVO: return
        if self.modo_reinicio == REINICIO_ESPERANDO:
            if hay_objetos:
                self.frames_sin_objeto = 0
                self.t_escena_vacia = None
                self.barra_progreso.set(0)
                self.lbl_reinicio_estado.configure(text='Retira las pelotas del campo de vision...', text_color='#e67e22')
            else:
                self.frames_sin_objeto += 1
                if self.frames_sin_objeto >= self.FRAMES_VACIA_MIN:
                    if self.t_escena_vacia is None: self.t_escena_vacia = time.time()
                    elapsed = time.time() - self.t_escena_vacia
                    progreso = min(1.0, elapsed / TIEMPO_REINICIO_SEG)
                    self.barra_progreso.set(progreso)
                    self.lbl_reinicio_estado.configure(text=f'Sin pelotas - reiniciando...', text_color='#2ecc71')
                    if elapsed >= TIEMPO_REINICIO_SEG: self._ejecutar_reinicio()
                else:
                    self.lbl_reinicio_estado.configure(text='Esperando escena vacia...', text_color='#f39c12')

    def _ejecutar_reinicio(self):
        self.modo_reinicio = REINICIO_LISTO
        self.barra_progreso.set(1.0)
        self.huellas_memoria = {'Rojo': [], 'Blanco': [], 'Negro': []}
        self.ultimo_intento = {}
        self.temporizadores = {'Rojo': 0.0, 'Blanco': 0.0, 'Negro': 0.0}
        for tr in self.trackers.values(): tr.reiniciar()
        self.after(0, self._agregar_separador_reinicio)
        self.btn_reinicio.configure(text='Reinicio', fg_color='#8e44ad', hover_color='#6c3483')
        self.lbl_reinicio_estado.configure(text='Memoria reseteada.', text_color='#2ecc71')
        def volver_normal():
            self.modo_reinicio = REINICIO_INACTIVO
            self.barra_progreso.set(0)
            self.lbl_reinicio_estado.configure(text='')
        self.after(3000, volver_normal)

    def _agregar_separador_reinicio(self):
        hora = datetime.datetime.now().strftime('%H:%M:%S')
        sep = ctk.CTkFrame(self.lista_scroll, fg_color='#2d1f4a', corner_radius=6, height=28)
        sep.pack(fill='x', pady=5, padx=2)
        sep.pack_propagate(False)
        ctk.CTkLabel(sep, text=f'-- REINICIO {hora} --', font=('Helvetica', 9, 'bold'), text_color='#9b59b6').pack(expand=True)
        self.lista_scroll._parent_canvas.yview_moveto(1.0)

    def _actualizar_barra_reinicio(self):
        if self.modo_reinicio == REINICIO_ESPERANDO and self.t_escena_vacia is None:
            self.barra_progreso.configure(progress_color='#8e44ad')
            self.barra_progreso.set(time.time() % 1.0)
            self.after(80, self._actualizar_barra_reinicio)
        else:
            self.barra_progreso.configure(progress_color='#f39c12')

    COLORES_UI = {'Rojo': ('#ff4d4d', 'Roja'), 'Blanco': ('#dddddd', 'Blanca'), 'Negro': ('#888888', 'Negra')}

    def on_objeto_detectado(self, nombre, x_norm, y_norm, z_est, huella):
        ahora = time.time()
        if ahora - self.ultimo_intento.get(nombre, 0.0) < self.COOLDOWN_INTENTO:
            return
        self.ultimo_intento[nombre] = ahora

        # LIMITAR A 2 REGISTROS MAXIMO POR OBJETO/COLOR
        num_color = sum(1 for o in self.historial_objetos if o['color'] == nombre)
        if num_color >= self.MAX_REGISTROS_POR_COLOR:
            return # Se ignora, ya hay 2 registrados.

        mem = self.huellas_memoria.setdefault(nombre, [])
        if huella is not None and es_mismo_objeto(huella, mem):
            return
        if huella is not None:
            mem.append(huella)

        total_memoria = sum(len(v) for v in self.huellas_memoria.values())
        entrada = {'color': nombre, 'num': num_color + 1, 'x': x_norm, 'y': y_norm, 'z': z_est,
                   'hora': datetime.datetime.now().strftime('%H:%M:%S')}
        self.historial_objetos.append(entrada)
        self.after(0, lambda e=entrada, tm=total_memoria: self._agregar_fila(e, tm))

    def _agregar_fila(self, entrada, total_memoria):
        color_hex, etiqueta = self.COLORES_UI.get(entrada['color'], ('#aaaaaa', entrada['color']))
        fila = ctk.CTkFrame(self.lista_scroll, fg_color='#242b42', corner_radius=8, border_width=1, border_color='#3a4460')
        fila.pack(fill='x', pady=3, padx=2)
        barra = ctk.CTkFrame(fila, fg_color=color_hex, width=5, corner_radius=4)
        barra.pack(side='left', fill='y', padx=(3, 6), pady=4)
        cont = ctk.CTkFrame(fila, fg_color='transparent')
        cont.pack(side='left', fill='both', expand=True, pady=4)
        ctk.CTkLabel(cont, text=f'Pelota {etiqueta} #{entrada["num"]}', font=('Helvetica', 12, 'bold'), text_color=color_hex, anchor='w').pack(anchor='w')
        ctk.CTkLabel(cont, text=f'X:{entrada["x"]:+.2f} Y:{entrada["y"]:+.2f} Z:{entrada["z"]}m', font=('Courier', 9), text_color='#6a7a9a', anchor='w').pack(anchor='w')
        ctk.CTkLabel(cont, text=entrada['hora'], font=('Helvetica', 9), text_color='#4a5a72', anchor='w').pack(anchor='w')
        self.lbl_conteo.configure(text=f'Total: {len(self.historial_objetos)}  |  En memoria: {total_memoria}')
        self.lista_scroll._parent_canvas.yview_moveto(1.0)

    def _limpiar_todo(self):
        for w in self.lista_scroll.winfo_children(): w.destroy()
        self.historial_objetos.clear()
        self.huellas_memoria = {'Rojo': [], 'Blanco': [], 'Negro': []}
        self.ultimo_intento = {}
        self.temporizadores = {'Rojo': 0.0, 'Blanco': 0.0, 'Negro': 0.0}
        for tr in self.trackers.values(): tr.reiniciar()
        self.lbl_conteo.configure(text='Total: 0  |  En memoria: 0')

    def _accion_buscar(self):
        self.btn_detectar.configure(text='Buscando hardware...', state='disabled')
        self.update()
        for w in self.frame_lista.winfo_children(): w.destroy()
        camaras = detectar_camaras_sistema()
        if camaras:
            self.lbl_estado.configure(text=f'Se encontraron {len(camaras)} camara(s)', text_color='#2ecc71')
            for idx, etiqueta in camaras:
                ctk.CTkRadioButton(self.frame_lista, text=etiqueta, variable=self.indice_sel, value=str(idx), font=('Helvetica', 13)).pack(anchor='w', pady=5, padx=10)
            self.indice_sel.set(str(camaras[0][0]))
            self.lbl_pie.configure(text='Selecciona la camara y presiona Iniciar')
            self.btn_iniciar.configure(state='normal')
        else:
            self.lbl_estado.configure(text='No se detecto hardware de video', text_color='#e74c3c')
            self.lbl_pie.configure(text='Asegurate de que la camara este conectada')
            self.btn_iniciar.configure(state='disabled')
        self.btn_detectar.configure(text='DETECTAR CAMARAS', state='normal')

    def _iniciar_camara(self):
        indice = int(self.indice_sel.get())
        if indice == -1: return
        self.btn_detectar.pack_forget()
        self.lbl_estado.pack_forget()
        self.lbl_lista_tit.pack_forget()
        self.frame_lista.pack_forget()
        self.lbl_pie.pack_forget()
        self.btn_iniciar.pack_forget()
        modo = 'YOLO (Mallas)' if self.yolo_model else 'HoughCircles'
        self.lbl_titulo.configure(text=f'MONITOR EN VIVO  -  V0.5  [{modo}]')
        self.frame_monitor.pack(expand=True, fill='both')
        self.cap = cv2.VideoCapture(indice, cv2.CAP_DSHOW)
        self.escaneando = False
        self.btn_escaneo.configure(text='Iniciar Escaneo', fg_color='#f39c12', hover_color='#d68910')
        self._loop_video()

    def _toggle_escaneo(self):
        self.escaneando = not self.escaneando
        if self.escaneando:
            self.btn_escaneo.configure(text='Pausar Escaneo', fg_color='#d35400', hover_color='#a04000')
            self.btn_reinicio.configure(state='normal')
            self.temporizadores = {'Rojo': 0.0, 'Blanco': 0.0, 'Negro': 0.0}
            for tr in self.trackers.values(): tr.reiniciar()
        else:
            self.btn_escaneo.configure(text='Reanudar Escaneo', fg_color='#f39c12', hover_color='#d68910')
            self.btn_reinicio.configure(state='disabled')
            if self.modo_reinicio != REINICIO_INACTIVO: self._cancelar_reinicio()

    def _loop_video(self):
        if not (self.cap and self.cap.isOpened()): return
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            hay_objetos = False
            if self.escaneando:
                frame, hay_objetos = procesar_frame_vision(frame, self.temporizadores, self.trackers, callback_objeto=self.on_objeto_detectado, yolo_model=self.yolo_model)
                self._procesar_estado_reinicio(hay_objetos)
            img = ctk.CTkImage(light_image=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), size=(920, 630))
            self.lbl_video.configure(image=img)
        self.after(15, self._loop_video)

    def _apagar_hardware(self):
        if self.cap:
            self.cap.release()
            self.cap = None
        if self.modo_reinicio != REINICIO_INACTIVO: self._cancelar_reinicio()
        self.escaneando = False
        self.frame_monitor.pack_forget()
        self.lbl_titulo.configure(text='CONFIGURACION DE CAMARA')
        self.btn_detectar.pack(pady=10)
        self.lbl_estado.pack(pady=(0, 20))
        self.lbl_lista_tit.pack(anchor='w', padx=150)
        self.frame_lista.pack(pady=5, padx=150)
        self.lbl_pie.pack(anchor='w', padx=150)
        self.btn_iniciar.pack(pady=(40, 0))

    def destroy(self):
        self._apagar_hardware()
        super().destroy()

print('Celda 3 V0.5: UI lista. (Registros limitados a 2 por objeto)')

Celda 3 V0.5: UI lista. (Registros limitados a 2 por objeto)


In [16]:
# Celda 4 - Entrenamiento YOLOv8  [V0.5]
from ultralytics import YOLO
from pathlib import Path

def train_model():
    BASE_DIR     = Path().absolute()
    DATASET_YAML = BASE_DIR / 'dataset.yaml'
    if not DATASET_YAML.exists():
        print(f'No se encontro: {DATASET_YAML}')
        return
    print('Iniciando entrenamiento ...')
    model = YOLO('yolov8n-seg.pt')
    model.train(task='segment', data=str(DATASET_YAML), epochs=50, imgsz=640, batch=16, project=str(BASE_DIR / 'models'), name='entrenamiento_pelotas', device='cpu')
    print('Listo. Modelo guardado en models/entrenamiento_pelotas/weights/best.pt')

# train_model()

In [17]:
# Celda 5 - Optimizador de Visión con DEAP [V0.5]
# Este algoritmo genetico evoluciona una regla matematica (arbol GP)
# para calcular el HOUGH_PARAM2 optimo basado en la iluminacion del entorno.

NUM_FASES       = 3
GEN_POR_FASE    = 5
TAM_POBLACION   = 40
TAM_ELITE       = 5
MAX_TREE_HEIGHT = 5
NUM_INSTANCIAS  = 10

class EntornoVision:
    def __init__(self, id_entorno, brillo, contraste, ruido, param2_ideal):
        self.id = id_entorno
        self.brillo = brillo
        self.contraste = contraste
        self.ruido = ruido
        self.param2_ideal = param2_ideal

def div_segura(izq, der):
    return izq / der if abs(der) > 1e-6 else 1.0

pset = gp.PrimitiveSet("MAIN", 3)
pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(div_segura,   2)
pset.renameArguments(ARG0='Brillo', ARG1='Contraste', ARG2='Ruido')

if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,)) # Minimizamos error
if not hasattr(creator, "IndividualGP"):
    creator.create("IndividualGP", gp.PrimitiveTree, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)
toolbox.register("individual", tools.initIterate, creator.IndividualGP, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("compile", gp.compile, pset=pset)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", gp.cxOnePoint)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=MAX_TREE_HEIGHT))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=MAX_TREE_HEIGHT))

def evaluar_regla(individuo, entornos):
    try:
        funcion = toolbox.compile(expr=individuo)
    except Exception:
        return (float('inf'),)
    errores = []
    for ent in entornos:
        try:
            prediccion = funcion(ent.brillo, ent.contraste, ent.ruido)
            errores.append(abs(prediccion - ent.param2_ideal))
        except:
            errores.append(100.0)
    penalizacion = len(individuo) * 0.1
    return (np.mean(errores) + penalizacion,)

def generar_entornos_simulados(num=NUM_INSTANCIAS):
    entornos = []
    for i in range(num):
        b = random.uniform(50, 200)
        c = random.uniform(0.5, 2.0)
        r = random.uniform(0.0, 5.0)
        p2_ideal = 40 + (b - 100)*0.1 - (c - 1.0)*5 + r*2 # Formula oculta a descubrir
        entornos.append(EntornoVision(f"E_{i}", b, c, r, p2_ideal))
    return entornos

def simular_optimizacion_vision():
    print("Iniciando Optimizador Evolutivo de Parametros de Vision...")
    entornos = generar_entornos_simulados()
    toolbox.register("evaluate", evaluar_regla, entornos=entornos)
    
    poblacion = toolbox.population(n=TAM_POBLACION)
    salon_fama = tools.HallOfFame(TAM_ELITE)
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("Promedio Error", np.mean)
    stats.register("Min Error", np.min)
    
    algorithms.eaSimple(poblacion, toolbox, cxpb=0.7, mutpb=0.2, ngen=GEN_POR_FASE,
                        stats=stats, halloffame=salon_fama, verbose=True)
    
    mejor = salon_fama[0]
    print("\n=== Mejor Regla Generada ===")
    print(f"HOUGH_PARAM2 = {mejor}")
    print(f"Error promedio: {mejor.fitness.values[0]:.4f}")

# Descomenta la linea de abajo para correr la simulacion DEAP:
# simular_optimizacion_vision()

In [18]:
# Celda 6 - Punto de entrada  [V0.5]
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()

Sin modelo YOLO. Modo HoughCircles activo.
